In [1]:
# %matplotlib qt
from matplotlib import cm
import matplotlib.pyplot as plt
import mne
from neuronol_power import psd_M4, _integrate_psd
from neuronol_signalprocessing import hilbert_spectra_from_raw
import numpy as np
import os
import pandas as pd
import time

# FOOOF
from fooof import FOOOFGroup, FOOOF
from fooof.plts.spectra import plot_spectrum
from fooof.bands import Bands
from fooof.analysis import get_band_peak_fg, get_band_peak_fm

In [2]:
def getAllFOOOFFeatures(cropped_raw, params, freq_bands, fooof_params, verbose=False):
    '''
    Calculate all required spectral power features for the given mne.Raw object.
    '''

    # Given
    freq_range = [0.5, 40]

    # Calculate power spectra
    psds_mean_epochs_M4, freqs = psd_M4(params, cropped_raw, plot_psd_epochs=None)
    mean_psd = np.mean(psds_mean_epochs_M4, axis=0)

    # Initialize a FOOOFGroup object, with desired settings (if any)
    # fm = FOOOF(peak_width_limits=[2, 12], max_n_peaks=6, min_peak_height=0, peak_threshold=2, aperiodic_mode='fixed')
    fm = FOOOF(
        peak_width_limits=fooof_params['peak_width_limits'],
        max_n_peaks=fooof_params['max_n_peaks'],
        min_peak_height=fooof_params['min_peak_height'],
        peak_threshold=fooof_params['peak_threshold'],
        aperiodic_mode=fooof_params['aperiodic_mode'],
    )

    # Fit the power spectrum model
    fm.fit(freqs, mean_psd, freq_range)

    # Get features F25-27, 40-41
    F = {str(i + 1): ('', '') for i in range(50)}
    for ftr_num, (band_lbl, band_def) in zip(['25', '26', '27', '41'], freq_bands):

        # Get the power values for the current band
        central_freq, amp, bandwidth = get_band_peak_fm(fm, band_def)
        mean_band_power = amp * (bandwidth / 2) * np.sqrt(2 * np.pi)    # net_area = net_amp * std_dev * sqrt(2 * pi)
        if np.isnan(mean_band_power): mean_band_power = 0   # if band power is nan => power is 0
        if verbose: print("F%s: Absolute power in %s band (%0.1f-%0.1f Hz) =" % (ftr_num, band_lbl, band_def[0], band_def[1]), mean_band_power)
        F[ftr_num] = (mean_band_power, 'TODO')

        if band_lbl == 'Range':
            domfreq = central_freq
            F['40'] = (domfreq, 'Hz')
            if verbose: print("F40: Dominant frequency over studied range (%0.1f-%0.1f Hz) = %0.3f Hz" % (band_def[0], band_def[1], domfreq))
    return F

def relativeAperiodicComponentFOOOF(spec, freqs, freq_range, fooof_params, verbose=None):
    """
    Evaluates aperiodic component of the given spectrum `spec` using FOOOF and
    calculates the ratio of AUC of aperiodic component and full spectrum.
    """

    # Initialize a FOOOFGroup object, with desired settings (if any)
    fm = FOOOF(
        peak_width_limits=fooof_params['peak_width_limits'],
        max_n_peaks=fooof_params['max_n_peaks'],
        min_peak_height=fooof_params['min_peak_height'],
        peak_threshold=fooof_params['peak_threshold'],
        aperiodic_mode=fooof_params['aperiodic_mode'],
    )

    # Fit the power spectrum model in the given `freq_range` and crop spectrum after fitting
    fm.fit(freqs, spec, freq_range)
    inds_range = [(np.abs(freqs - freq_range[0])).argmin(), (np.abs(freqs - freq_range[1])).argmin()]
    inds = np.arange(inds_range[0], inds_range[1]+1)
    spec = spec[inds]; freqs = freqs[inds]

    # Extract aperiodic params of the spectrum
    chi = fm.get_params('aperiodic_params', 'exponent')
    b = fm.get_params('aperiodic_params', 'offset')
    if fooof_params['aperiodic_mode'] == 'fixed':
        k = 0
    elif fooof_params['aperiodic_mode'] == 'knee':
        k = fm.get_params('aperiodic_params', 'knee')
    else:
        raise ValueError("The value of `fooof_params['aperiodic_mode']` can only be 'fixed' or 'knee'")
    if verbose and verbose is not None:
        print("Aperiodic parameters are: Offset (b) = %0.4f; Exponent (chi) = %0.4f; Knee (k) = %0.4f" % (b, chi, k))

    # Calculate aperiodic component of the spectrum
    spec_aper = (10 ** b) * 1 / (k + np.power(freqs, chi))

    # Calculate relative AUC of aperiodic to whole mean spectrum
    spec_int = _integrate_psd(spec, freqs, freq_range)
    spec_aper_int = _integrate_psd(spec_aper, freqs, freq_range)
    rltv_aper = spec_aper_int / spec_int
    if verbose and verbose is not None:
        print("\n[%s] Relative AUC of aperiodic to whole spectrum = %0.3f%%\n" % (spec_mode.upper(), rltv_aper * 100))

    return rltv_aper, b, chi, k

## Given

In [3]:
# Path to files
dir_data = os.path.expanduser('~/data/wash-u/preprocessed/v2')
dir_ftrs = os.path.expanduser('~/research/results/wash-u/features/fft-fooof')
dir_results = os.path.expanduser('~/research/results/wash-u/')

# Basic parameters
params = {}
params['size_epoch'] = 1.2

# Freq bands
freq_bands = Bands({
    'Delta+Theta': [1.3, 7.5],
    'Alpha': [7.5, 13],
    'Beta': [13, 35],
    'Range': [1.3, 35],
})

# FOOOF Params
fooof_params = {
    'peak_width_limits': [2, 12],
    'max_n_peaks': 6,
    'min_peak_height': 0,
    'peak_threshold': 2,
    'aperiodic_mode': 'fixed'
}
freq_range = [50 / 60, 40]

# Derive all required features from all EEG files

In [ ]:
# Start clocking execution
start_time = time.time()

truncated_files = []
fnames = [f for f in os.listdir(dir_data) if f[-4:] == '.fif']  # search for all FIF files in `dir_data`
for i_file, fname_raw in enumerate(fnames):

    # Read preprocessed EEG data
    fpath_raw = os.path.join(dir_data, fname_raw)
    raw = mne.io.read_raw_fif(fpath_raw)
    params['sfreq'] = np.floor(raw.info['sfreq'])

    # Find intervals for eyes closed resting state
    ec_end = [raw.annotations.onset[i] for i, ant_name in
                enumerate(raw.annotations.description) if ant_name in ('1', '3')]

    ec_intvl = {}
    ec_intvl['EC1'] = np.floor([ec_end[0] - 63, ec_end[0] - 3])
    ec_intvl['EC2'] = np.floor([ec_end[1] - 63, ec_end[1] - 3])

    for ec in ec_intvl:
        print('\nEvaluating Condition-%s...\n' % ec)

        tmin, tmax = ec_intvl[ec]
        if tmin > 0:
            cropped_raw = raw.copy().crop(tmin=tmin, tmax=tmax)
            params['T_max'] = cropped_raw.times[-1] - cropped_raw.times[0]
            features = getAllFOOOFFeatures(cropped_raw, params, freq_bands, fooof_params, verbose=False)

            # Export features in CSV format as a DataFrame
            df = pd.DataFrame(features).T
            df.columns = ['Values', 'Units']
            fname_ftrs = fname_raw[:15] + '_cond=%s_Tmax=%.3f.csv' % (ec, params['T_max'])
            # fname_ftrs = fname_raw[:15] + '_cond=' + ec + '.csv'
            fpath_ftrs = os.path.join(dir_ftrs, fname_ftrs)
            print(fpath_ftrs)
            df.to_csv(fpath_ftrs)
        else:
            truncated_files.append((fname_raw, ec))

# End clocking execution and display result
execution_time = (time.time() - start_time)
print('Execution time in seconds:', execution_time)

# Calculate aperiodic AUD and model fit parameters for FFT/HHT spectra

In [6]:
spec_modes = ['fft', 'hht']
truncated_files = []
fnames = [f for f in os.listdir(dir_data) if f[-4:] == '.fif']  # search for all FIF files in `dir_data`

results = {
    'FilePathRaw': [],
    'EyesClosedEpoch': [],
    'RelativeAperiodicity_FFT': [],
    'RelativeAperiodicity_HHT': [],
    'Offset_FFT': [],
    'Offset_HHT': [],
    'Exponent_FFT': [],
    'Exponent_HHT': [],
    'Knee_FFT': [],
    'Knee_HHT': [],
}

for i_file, fname_raw in enumerate(fnames):

    # Read preprocessed EEG data
    fpath_raw = os.path.join(dir_data, fname_raw)
    raw = mne.io.read_raw_fif(fpath_raw)

    # Find intervals for eyes closed resting state
    ec_end = [raw.annotations.onset[i] for i, ant_name in
                enumerate(raw.annotations.description) if ant_name in ('1', '3')]
    ec_intvl = {}
    ec_intvl['EC1'] = np.floor([ec_end[0] - 63, ec_end[0] - 3])
    ec_intvl['EC2'] = np.floor([ec_end[1] - 63, ec_end[1] - 3])

    for ec in ec_intvl:

        tmin, tmax = ec_intvl[ec]
        if tmin > 0:
            # Populate results dict
            results['FilePathRaw'].append(fpath_raw)
            results['EyesClosedEpoch'].append(ec)

            # Crop raw
            cropped_raw = raw.copy().crop(tmin=tmin, tmax=tmax)

            ## Calculate power spectra
            mean_specs = {}
            for spec_mode in spec_modes:
            # for spec_mode in ['fft']:
                if spec_mode == 'fft':
                    params['sfreq'] = np.floor(raw.info['sfreq'])
                    params['T_max'] = cropped_raw.times[-1] - cropped_raw.times[0]
                    specs_mean_epochs, freqs = psd_M4(params, cropped_raw, plot_psd_epochs=None)
                elif spec_mode == 'hht':
                    # specs_mean_epochs, freqs = psd_M4(params, cropped_raw, plot_psd_epochs=None)
                    specs_mean_epochs, freqs = hilbert_spectra_from_raw(cropped_raw, compute_power_spec=True)
                else:
                    raise ValueError("`spec_mode` can only be 'fft' or 'hht'")
                mean_spec = np.mean(specs_mean_epochs, axis=0)

                # Calculate relative AUC of aperiodic to whole mean spectrum
                rel_aper, offset, exp, knee = relativeAperiodicComponentFOOOF(mean_spec, freqs, freq_range, fooof_params, verbose=False)

                # Populate results dict
                results['RelativeAperiodicity_' + spec_mode.upper()].append(rel_aper)
                results['Offset_' + spec_mode.upper()].append(offset)
                results['Exponent_' + spec_mode.upper()].append(exp)
                results['Knee_' + spec_mode.upper()].append(knee)
        else:
            truncated_files.append((fname_raw, ec))

df_results = pd.DataFrame.from_dict(results)
fname_aperiodicity_results = "aperiodicity_comparison.csv"
fpath_results = os.path.join(dir_results, 'aperiodicity', fname_aperiodicity_results)
df_results.to_csv(fpath_results, index=False)
df_results

Opening raw data file C:\Users\chholakp2/data/wash-u/preprocessed/v2\7005_4_rest1_ec_ica_ssp_eeg.fif...
Isotrak not found
    Read a total of 2 projection items:
        ECG-eeg--0.200-0.400-PCA-01 (1 x 30) active
        ECG-eeg--0.200-0.400-PCA-02 (1 x 30) active
    Range : 0 ... 155879 =      0.000 ...   311.758 secs
Ready.
A total of 60 seconds of raw EEG data has been studied.
Fs = 500.0
Each epoch has 1.2 seconds of EEG data.
Shape of raw_data = (30, 30000)
Total number of epochs studied = 50
Shape of epochs = (50, 30, 600)
Shape of psds = (50, 30, 299)
A total of 60 seconds of raw EEG data has been studied.
Fs = 500.0
Each epoch has 1.2 seconds of EEG data.
Shape of raw_data = (30, 30000)
Total number of epochs studied = 50
Shape of epochs = (50, 30, 600)
Shape of psds = (50, 30, 299)
Opening raw data file C:\Users\chholakp2/data/wash-u/preprocessed/v2\7005_4_rest2_ec_ica_ssp_eeg.fif...
Isotrak not found
    Read a total of 2 projection items:
        ECG-eeg--0.200-0.400-PCA-0

,FilePathRaw,EyesClosedEpoch,RelativeAperiodicity_FFT,RelativeAperiodicity_HHT,Offset_FFT,Offset_HHT,Exponent_FFT,Exponent_HHT,Knee_FFT,Knee_HHT
0,C:\Users\chholakp2/data/wash-u/preprocessed/v2...,EC1,0.435353,0.417177,3.469985,-0.358249,1.632558,1.770016,0,0
1,C:\Users\chholakp2/data/wash-u/preprocessed/v2...,EC2,0.483814,0.452435,3.536177,-0.305455,1.654227,1.793012,0,0
2,C:\Users\chholakp2/data/wash-u/preprocessed/v2...,EC1,0.475105,0.397101,3.585260,-0.408711,1.630094,1.606541,0,0
3,C:\Users\chholakp2/data/wash-u/preprocessed/v2...,EC2,0.536839,0.476620,3.687541,-0.208967,1.799528,1.948214,0,0
4,C:\Users\chholakp2/data/wash-u/preprocessed/v2...,EC1,0.507075,0.511278,3.266028,-0.547182,1.578699,1.680962,0,0
...,...,...,...,...,...,...,...,...,...,...
407,C:\Users\chholakp2/data/wash-u/preprocessed/v2...,EC2,0.727888,0.854387,3.825558,0.083492,1.645251,1.793743,0,0
408,C:\Users\chholakp2/data/wash-u/preprocessed/v2...,EC1,0.664567,0.709901,3.467598,-0.335432,1.347158,1.426970,0,0
409,C:\Users\chholakp2/data/wash-u/preprocessed/v2...,EC2,0.638201,0.690476,3.483345,-0.305365,1.382614,1.492968,0,0
410,C:\Users\chholakp2/data/wash-u/preprocessed/v2...,EC1,0.524521,0.522775,3.559803,-0.325617,1.063369,1.087868,0,0


# Test fitting parameters

In [ ]:
## Load EEG data and calculate power spectra of all channels

truncated_files = []; psds = []
fnames = [f for f in os.listdir(dir_data) if f[-4:] == '.fif']  # search for all FIF files in `dir_data`
np.random.seed(0); np.random.shuffle(fnames)

for i_file, fname_raw in enumerate(fnames):

    # Read preprocessed EEG data
    fpath_raw = op.join(dir_data, fname_raw)
    raw = mne.io.read_raw_fif(fpath_raw)
    params['sfreq'] = np.floor(raw.info['sfreq'])

    # Find intervals for eyes closed resting state
    ec_end = [raw.annotations.onset[i] for i, ant_name in
                enumerate(raw.annotations.description) if ant_name in ('1', '3')]
    ec_intvl = {}
    ec_intvl['EC1'] = np.floor([ec_end[0] - 63, ec_end[0] - 3])
    ec_intvl['EC2'] = np.floor([ec_end[1] - 63, ec_end[1] - 3])

    for ec in ec_intvl:
        tmin, tmax = ec_intvl[ec]
        if tmin > 0:
            # Crop raw
            cropped_raw = raw.copy().crop(tmin=tmin, tmax=tmax)
            params['T_max'] = cropped_raw.times[-1] - cropped_raw.times[0]
            
            # Calculate power spectra
            psds_mean_epochs_M4, freqs = psd_M4(params, cropped_raw, plot_psd_epochs=None)
            mean_psd = np.mean(psds_mean_epochs_M4, axis=0)
            psds.append(mean_psd)
        else:
            truncated_files.append((fname_raw, ec))

In [ ]:
## Fitting FOOOF power spectrum model

#   Initialize a FOOOFGroup object, with desired settings (if any)
fg = FOOOFGroup(peak_width_limits=[2, 12], max_n_peaks=6, min_peak_height=0,
           peak_threshold=2, aperiodic_mode='fixed')

#   Define the frequency range to fit
freq_range = [0.5, 40]

#   Fit the power spectrum model across all channels
fg.fit(freqs, np.array(psds), freq_range)
fg.report()

In [ ]:
## Individual model exploration
# i_ch_worst = np.argmax(fg.get_params('error'))
i_ch_worst = 60
print(i_ch_worst)
fm = fg.get_fooof(i_ch_worst)
fm.report(plt_log=False)